# Workshop Robotics Lesson 6 - "Advanced Robotics Techniques to Explore"


## Introduction

Congratulations on reaching this point! Many of the topics in these lessons are complex, and the work can be challenging. The previous modules of this workshop provides a strong foundational understanding of how robots function, which you can build upon by exploring more advanced topics. In this lesson, we’ll summarize the advanced modules and share resources related them for application with your own robots.

The final section of this module showcases the construction of our new rover to test these advanced robotics techniques.

# Robotic Perception and Control(*Module 7*)

Advanced robotics requires more than just sending commands and assuming perfect conditions. Feedback control, state estimation, and sensor fusion are essential for autonomous systems that need to adapt to unpredictable environments.

In Lesson 3, we assumed a linear relationship between the voltage applied to our motors and the speed at which they operate. This was an example of feed-forward control, where we assume a consistent relationship between inputs and outputs. While this works under ideal conditions, real-world disturbances—such as driving up a hill—can introduce resistance, slowing the rover down. 

![Feed-forward control may not adequately handle disturbances](6-1.png)

# Feedback Control Systems

## Feedback vs. Feed-Forward Control

To counteract disturbances like inclines, we can use feedback control instead. Instead of assuming a fixed relationship between input and output, feedback control measures the actual speed of the rover using motor encoders and adjusts the control input accordingly. The system calculates an **error value**, which is the difference between the **desired velocity** and the **actual velocity**. The feedback controller then modifies the control input to correct this error. 

![Feedback control bases the control input on the difference between a desired output  and the actual output](6-2.png)

For example, if the rover starts climbing a hill and slows down, the error increases. The feedback controller detects this and increases power to the motors until the rover reaches the desired velocity again. This adaptability makes feedback control tolerant of external disturbances.

## What is a Feedback Controller?

A **feedback controller** is a mathematical function that takes the error value and determines an appropriate control adjustment. There are multiple ways to design and tune these controllers, but one of the most common and relatively simple approaches is **PID Control** (Proportional-Integral-Derivative Control).

While there are **PID libraries** available for Arduino, writing a basic PID controller from scratch can be a useful learning experience.

Why Use Feedback Control?
You might wonder if feedback control is worth the effort. For a simple rover, it might not make a major difference. However, feedback control becomes essential in more complex systems, such as:

- Self-balancing robots (inverted pendulums)
- Aircraft autopilot systems
- Satellite pointing systems
- Cruise control in vehicles

A great way to explore feedback control is by building an inverted-pendulum robot (similar to a Segway). 

![Example of an inverted-pendulum robot](6-3.png)

These robots are inherently unstable, but a well-tuned feedback controller can actively balance them in real time using an **IMU sensor** (such as the **ICM-20948** from **Module 5**). 



# State Estimation and Sensor Fusion

In Module 5, we explored the **Inertial Measurement Unit (IMU)** and how it provides motion-tracking data using accelerometers, gyroscopes, and magnetometers. However, each sensor has limitations:

- Gyroscopes provide precise angular velocity but suffer from gyroscopic drift, causing inaccuracies over time.
- Accelerometers measure angular position but are noisy and imprecise.

To improve accuracy, state estimation techniques, such as the Kalman filter, allow us to combine multiple sensor measurements to compensate for weaknesses and create more reliable data.

## Kalman Filtering for Motion Tracking

The **Kalman filter** is a mathematical technique used in **sensor fusion**. It works by continuously refining an estimate of a system’s state based on both sensor data and a predictive model.

Let's consider an inverted pendulum robot using an **IMU** to determine its angular position.

- Step 1: Prediction Step
    - The gyroscope provides the rate of change of the robot’s tilt angle.
    - By integrating the gyroscope readings, we estimate the current angle.
    - However, due to drift, this prediction accumulates error over time.

- Step 2: Update Step
    - The accelerometer measures the tilt angle using gravity as a reference.
    - The Kalman filter compares the **measured angle** (from the accelerometer) with the **predicted angle** (from the gyroscope).
    - If the **difference is small**, we trust the gyroscope more.
    - If the **difference is large**, we correct the gyroscope estimate using the accelerometer data.
    
This process repeats in **real-time**, correcting for **drift and smoothing out noise**, resulting in an **accurate angular position estimate**.

![Side profile of inverted pendulum robot with IMU axes labeled](6-9.png)

## Should You Look Into Module 7?

If you want to:

✔ **Improve the precision of your rover’s movements.**

✔ **Explore advanced control techniques, like PID tuning.**

✔ **Develop systems that adjust to real-world disturbances dynamically.**

✔ **Learn sensor fusion for better localization.**

✔ **Implement state estimation to reduce sensor drift.**

Then **Module 7 (Robotic Perception and Control)** is for you!

This module will delve deeper into:

- Motion stability and PID control.
- State estimation for reliable position tracking.
- Sensor fusion to create a smarter, more autonomous rover.

# Localization, Mapping, and Motion Planning  (*Module 8*)

In Module 4, we used a **time-of-flight sensor** to measure distances. However, by rotating the sensor and collecting multiple readings, we can map an **area** instead of just measuring a single point. This technique is used in **2D LIDAR** sensors, which can build a floor plan of the environment. 

![Time-of-flight sensor compared to a 2D lidar sensor](6-4.png)

## Mapping and Odometry

We have previously used wheel odometry (tracking wheel movement) to estimate the rover’s position. By combining **odometry** with a **time-of-flight sensor**, we can map obstacles in the environment relative to the rover's starting point.

For example, if the rover:

-Drives forward 1 meter and right 2 meters
-Detects an obstacle at 4 meters straight ahead
-Detects a second obstacle at 37° to the right at 5 meters

Using trigonometry, we can place these obstacles relative to the starting position. This allows the rover to build a simple map of its surroundings. 

![The rover starts, drives to position (2, 1), and sees two obstacles](6-5.png)

## Challenges in Odometry

Wheel odometry works well on solid surfaces but struggles on:

- Slippery terrain (e.g., ice, sand, or dirt)
- Uneven or bumpy surfaces

Even on ideal surfaces, **small wheel slips accumulate over time**, making odometry **increasingly inaccurate**. To correct these errors, we can use **Simultaneous Localization and Mapping (SLAM)** techniques.

SLAM allows the rover to:

- Continuously correct its estimated position using its environment
- Update its map dynamically based on new sensor data

For instance, if the rover turns **left 90°** and **drives forward**, its **odometry** may incorrectly estimate that it has traveled **2 meters forward**. However, if the mapped obstacles appear at **incorrect locations**, the rover can adjust its position estimate to realign with its map. 

![The rover fixes its odometry to align with the map](6-6.png)

This iterative process—combining mapping with localization corrections—is the foundation of SLAM. 

![Simulation of SLAM on differential drive rover](6-7.png)

## Path Planning with Costmaps

So far, our rover has navigated by reacting to obstacles. But when we have a map and can localize within it, we can plan a route rather than blindly moving forward.

One common approach to path planning is to:

- Divide the map into discrete grid cells (nodes)
- Assign a cost to each cell (e.g., high cost near obstacles)
- Find the lowest-cost path to the goal

This technique is called a costmap-based pathfinding system. 

![An example costmap generated from a SLAM simulation, colors nearby obstacles  represent the cost of entering those nodes](6-8.png)

## A* Pathfinding Algorithm

A widely used algorithm for planning paths through a **costmap** is *A **(A-Star)**. The algorithm works by evaluating the cost of moving through different nodes using the function:

$$f(n)=g(n)+h(n)$$

Where:

- $g(n)$ = cost to reach node n from the start

- $h(n)$ = estimated cost from node n to the goal

The algorithm expands paths **incrementally**, always choosing the **lowest-cost** option, until it finds the **shortest and safest** path to the goal.

This process builds a **tree of possible paths**, expanding outward from the rover’s starting position. Over time, the best path emerges, minimizing **travel distance and collision risks**.


## Should You Look Into Module 8?

If you are interested in:

✔ **Building a robot that can map its environment**

✔ **Developing real-time localization and tracking**

✔ **Implementing advanced path planning (e.g., A and costmap optimization)**

Then **Module 8 (Autonomous Navigation & SLAM)** is the next step! This module will focus on creating an autonomous robot that can:

- Map its environment using SLAM techniques
- Accurately determine its position in a dynamic environment
- Plan efficient paths and navigate obstacles intelligently

If your goal is to build robots that think and move independently, **Module 8** is a must!

# Computer Vision (*Module 9*)

## Giving Robots the Ability to See

For humans, vision is one of the most critical senses for understanding and interacting with the world. Similarly, advanced robotics systems benefit from computer vision, which enables robots to process and interpret images, recognize objects, and navigate environments effectively.

Robots that can “see” their environment unlock new possibilities for automation, navigation, and object interaction. Computer vision enables robots to process visual data, recognize objects, and make intelligent decisions based on what they perceive.

![Object Detection With Computer Vision](6-10.png)

## Convolutional Neural Networks (CNNs)

A fundamental concept in computer vision is convolution, a mathematical operation used to process and filter images. Convolutions help extract important features from an image before feeding it into a neural network, a type of artificial intelligence model. A basic intelligence model can be found for you to expand upon in module 9. This model has been trained using annotated images that establish boundaries of an object type.

When combined with neural networks, convolutions form a Convolutional Neural Network (**CNN**), a key technology in modern image recognition, object detection, and autonomous navigation. CNNs are widely used in robotics applications such as self-driving cars and facial recognition systems.

## How Do Robots “See”?

Unlike humans, robots don’t have built-in vision—they rely on cameras and sensors combined with powerful algorithms to interpret their surroundings. In this module, you will explore:
- **Object Detection** – Teaching a robot to recognize and classify objects in images
- **Dataset Annotation & Training** – Building a dataset, labeling images, and training a deep learning model
- **Live Inference & Real-Time Processing** – Running computer vision models on embedded devices like Raspberry Pi
    
## How Does Object Detection Work?

To recognize objects, a robot must first be trained using a Convolutional Neural Network (CNN)—a specialized deep learning model designed for image processing. CNNs detect patterns such as edges, shapes, and textures, gradually building up a complex understanding of what an object is.

In Module 9, you will learn how to:
- Use LabelMe to annotate images for training
- Train a YOLO (You Only Look Once) model for real-time object detection
- Convert and deploy a model for use on Raspberry Pi or other embedded systems

## Computer Vision Tools & Techniques

- LabelMe: A tool for drawing precise annotations on images to create training data
- YOLO Object Detection: A powerful AI model for real-time image recognition
- Raspberry Pi Camera + OpenCV: Capturing and processing live video feeds for robotic vision applications

## Should You Look Into Module 9? (Computer Vision)

By the end of Module 9, you will have trained your own object detection model and deployed it on a real system. This knowledge is essential for robotics applications like:

✔ **Obstacle Detection** – Helping robots navigate complex environments

✔ **Autonomous Sorting** – Allowing robots to recognize and categorize objects

✔ **AI-Assisted Navigation** – Enabling path planning based on visual data

If you want to give your robot vision, develop AI-powered object detection, and work with real-world image processing, then Module 9 is your next step!

This module will introduce you to:

- Using the Raspberry Pi Camera for live video feeds
- OpenCV for object detection, color tracking, and filtering
- Neural networks (CNNs) for advanced visual recognition
- Integrating vision with robotics for navigation and decision-making

Computer vision allows robots to perceive and interact with their environments in ways beyond simple sensors. If you want to work with **gesture recognition, lane detection, or even AI-driven navigation**, **Module 9** is your next step!

# Advanced Computer Vision for Autonomous Navigation (Module 10)

As robots become more autonomous, they must not only detect objects but also **understand their surroundings** and **make intelligent navigation decisions**. In Module 10, we will explore advanced computer vision techniques that integrate with our A* pathfinding algorithm and Simultaneous Localization and Mapping (SLAM).

![Hal9000](hal.jpg)

## What Will We Cover?

1. **Heuristic Assignment for Object Identification** – Enhancing the A* algorithm by classifying objects as obstacles or navigable space.
2. **Depth Estimation Using a Monocular Camera** – Estimating distances using a single camera to improve 3D perception.
3. **Computer Vision for SLAM** – Using visual data to help the robot track its position while mapping its environment.

## Why Does This Matter?

For a robot to navigate effectively, it must:

- **Recognize** obstacles and paths using computer vision.
- **Estimate** how far objects are without relying on expensive depth sensors.
- **Continuously** update its map while moving in an environment.

## Key Techniques in Module 10

## What You’ll Accomplish in Module 10

By the end of this module, you will have:

✔ Integrated computer vision-based obstacle detection into A* pathfinding.

✔ Implemented depth estimation to enhance navigation without additional sensors.

✔ Used a monocular camera for vision-based SLAM, improving localization.

If you're interested in teaching your robot to “see” the world in 3D, navigate using only a camera, and integrate AI-driven perception into real-time decision-making, then Module 10 is your next challenge!

# So You've Decided to Tackle an Advanced Module

We're excited for your determination but sadly the smaller rover you built previously will not hold up to the specifications of each of these modules. In this next section we will build the Advanced Rover for this workshop to give you additional practice on rover construction and allow you to progress through the advanced modules.

# **ADVANCED ROBOT CONSTRUCTION INSTRUCTIONS**

To implement any of the advanced topics covered in this module and progress to the next stages, you will need to upgrade the current workshop rover. This section will guide you through all the necessary modifications, including printing a new chassis, installing additional components, and making adjustments to the existing code. These modifications will provide the computational power and sensing capabilities required for advanced robotics techniques, such as **computer vision, SLAM, and real-time motion planning**.

## Step 1: Printing the New Chassis

### Why Do We Need a New Chassis?

The **Advanced Rover Chassis** is designed to accommodate additional hardware components, such as a Raspberry Pi, a camera module, and a compute battery, while maintaining the structural integrity needed for autonomous operations. The new chassis also repositions the motors, requiring a modified motor controller script (provided in the **Code Section** below).

### How to Print the Chassis

1. Access the Design:

    - Navigate to Onshape and search for “**COSGC Robotics Workshop Rover Chassis Advanced**”.
    - Right-click on the resulting document and select “**Copy Workspace**”.

2. Export the Chassis for 3D Printing:

    - Locate the **chassis tab** within Onshape.
    - Right-click on it and select “**Export**”.
    - Download the file in **STL format**.

3. Prepare for Printing:

    - Import the STL file into a slicer software (such as Cura or PrusaSlicer).
    - Choose the correct material and settings based on your printer’s capabilities.
    - Print the chassis and ensure structural integrity before proceeding.


## Step 2: Setting Up New Components

The **Advanced Rover** introduces several new components to support computer vision and advanced control functions. Below is an overview of the added components and their roles:

### New Components and Their Functions

1. Raspberry Pi 5
    - Acts as the onboard **high-performance computer**, enabling real-time computer vision, SLAM, and motion planning.
    - Runs **Linux** and handles complex computational tasks that an **Arduino** alone **cannot**.
    
![raspberry pi 5](RasPi5.jpg)

2. Micro SD Card
    - **Stores** the operating system (Raspberry Pi OS) and project files.
    - **Minimum 32GB** recommended for storage space.
3. Raspberry Pi Power Supply
    - A **dedicated power adapter** that supplies stable voltage to the Pi.
    
![Raspberry Pi Power Supply](powersupply.jpg)

4. Pi Camera Module 3
    - Provides **visual input** for tasks such as object detection, depth estimation, and SLAM.
    - Attaches directly to the Pi via the **gold-colored ribbon cable**.
    
![Pi Camera Module 3](camera.jpg)

5. Compute Battery (USB Power Bank)
    - **Portable power source** for the Pi.
    - Must provide **at least 3A at 5V** to ensure stable operation.
    
![Compute Battery](battery.jpg)

6. Miscellaneous Accessories
    - Mini HDMI cable, display, mouse, and keyboard – **Needed for initial Raspberry Pi setup**.

### Setting Up the Raspberry Pi

1. Flash the OS onto the SD Card

    - Use **Raspberry Pi Imager** to install **Raspberry Pi OS** on the micro SD card.
    - Insert the SD card into the Pi.

2. Boot the Pi and Complete Setup

    - Connect the **power adapter, keyboard, mouse, and display**.
    - Power on the Pi and follow the initial setup instructions.

3. Connect and Mount the Camera

    - Attach the **Pi Camera Module 3** to the **dedicated camera port**.
    - Ensure the **gold contact strips** are facing the correct direction.
    - Mount the camera securely to the chassis using screws.


## Step 3: Transferring Old Components & Assembling the New Robot

Now that we have prepared the chassis and installed the necessary components, we can assemble the new Advanced Rover by transferring and installing the old and new parts.

1. Removing Components from the Original Rover
    - **Unscrew** the **motor housings** and **caster ball housing** from the old chassis.
    - **Carefully** peel off the **breadboards**.
2. Installing the Motors and Breadboards
    - **Attach the caster housing and motors** to the new chassis.
    - **Reposition the breadboards** as per the reference images below.
3. Installing the Advanced Components
    - **Place the compute battery** between the **two motor wings**.
    - **Position the Raspberry Pi** against the compute battery, **aligning with the higher side** of the chassis wall.
    - **Secure the Pi Camera Module** using screws, ensuring it **faces forward** for navigation tasks.
    
![advanced rover](rover.png)
    
### Important: Due to the reoriented motors, you must update the motor driver code. Use new_motor_controller.ino from the Code Section below.

## Step 4: Implementing Test Code

Before proceeding with advanced tasks, we must verify that all components function correctly. The following tests ensure that the Pi and Arduino communicate properly, the **camera is working**, and the **motor controller is functional**.

**Test #1**: **Bi-Directional Communication (Pi ⇄ Arduino via USB)**

- **Purpose**: Ensures that the Pi can send and receive messages from the Arduino.
- **Required Software**: Python (with pyserial package), Arduino IDE.

**Instructions**

1. **Connect the Arduino** to the Pi using a **USB cable**.
2. **Upload** communicate.ino to the Arduino using the Arduino IDE.
3. **Save** communicate.py to the Pi and run it with:

```
python communicate.py
```

4. **Expected Output**:
```
Connected to Arduino
Sent to Arduino: Hello World
Received from Arduino: Echo: Hello World
```

**Test #2**: **Camera Functionality**

- **Purpose**: Ensures the Pi Camera is working correctly.
- **Required Software**: Python (picamera2 package).

**Instructions**

1. **Create** camera.py on the Pi.
2. **Run the script**:
```
python camera.py
```
3. **Expected Outcome**: A live camera feed should appear for **5 seconds** before closing.



## Step 5: Troubleshooting

If you encounter **issues during setup**, use the following **troubleshooting steps**:

| **Issue**                           | **Solution**                                      |
|--------------------------------------|--------------------------------------------------|
| Arduino IDE not installed           | `sudo apt install arduino`                      |
| Pyserial missing                     | `sudo apt install python3-serial`               |
| Picamera2 missing                    | `sudo apt install python3-picamera2`            |
| Permission denied (serial access)    | `sudo usermod -aG dialout <username>`           |
| Permission denied (camera access)    | `sudo usermod -aG video <username>`             |


## Step 6: Updated Code for the Advanced Rover

### New Motor Controller (new_motor_controller.ino)

This updated script accommodates the new chassis layout, ensuring proper motor control.

```
const float r = 0.030;
const float L = 0.2286;  // Updated width separating the drive wheels

const int R1 = 5, R2 = 4, pwmR = 3;
const int L1 = 8, L2 = 7, pwmL = 6;

void setup() {
  pinMode(R1, OUTPUT); pinMode(R2, OUTPUT); pinMode(pwmR, OUTPUT);
  pinMode(L1, OUTPUT); pinMode(L2, OUTPUT); pinMode(pwmL, OUTPUT);
}

void loop() {
  motor_controller(0.346, 0); delay(1000);
  motor_controller(-0.346, 0); delay(1000);
  motor_controller(0, 4.73); delay(1000);
  motor_controller(0, -4.73); delay(1000);
}

void motor_controller(float v, float w) {
  float dphi_L = -(v / r) + (L * w) / (2 * r);
  float dphi_R = -(v / r) - (L * w) / (2 * r);
  dphi_L = constrain(dphi_L, -11.52, 11.52);
  dphi_R = constrain(dphi_R, -11.52, 11.52);
  drive(map(dphi_L, -11.52, 11.52, -255, 255), map(dphi_R, -11.52, 11.52, -255, 255));
}
```

# Conclusion

With the Advanced Rover Chassis and new components installed, your robot is now prepared to tackle **computer vision, SLAM, and autonomous navigation**. The upgraded system provides the necessary computational power and sensor integration to support Modules 7-10. Let’s move forward into advanced robotics!